In [16]:


"""
大致流程
1. 先设置随机种子,选择设备
2. 定义模型,损失函数,优化器和metric
3. 调用 train_and_evaluate 进行训练和验证


    - nn.Model 负责组织模型参数和前向计算
    - loss Function  负责把预测和目标变成一个标量损失
    - optim.Optimizer 负责根据梯度更新参数
    - model.state_dict()  负责保存模块和优化器的状态
"""

import random
import dnnlpy
import torch
import numpy as np
import torch.accelerator as accl
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as utils
import torchvision.datasets as datasets
import torchvision.transforms.v2 as v2
from torch import Tensor
from torch.types import Device
from torchmetrics import Metric
from torchmetrics.classification import MulticlassAccuracy

print("PyTorch  version:", torch.__version__)

PyTorch  version: 2.13.0+cpu


In [17]:
"""
固定随机数种子
"""


def set_seed(seed: int | None = None, *, deterministic: bool = False, benchmark: bool = False,
             warn_only: bool = False) -> torch.Generator:
    random.seed(seed)
    np.random.seed(seed)
    if seed is not None:
        torch_rng = torch.manual_seed(seed)
    else:
        torch_rng = torch.default_generator

    torch.use_deterministic_algorithms(deterministic, warn_only=warn_only)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = benchmark

    return torch_rng


torch_rng = set_seed(42)
print(torch_rng)



In [18]:
"""
选择计算的设备
    - 模型在哪里数据就要在哪里
"""


def get_default_device() -> torch.device:
    device = accl.current_accelerator(check_available=True)
    if device is not None:
        return device
    return torch.device("cpu")


device = get_default_device()
print("Using device:", device)



Using device: cpu


In [19]:
"""
准备很小的分类任务
"""

root = dnnlpy.get_data_root()
transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
ds_rng = torch.Generator().manual_seed(42)

train_ds = datasets.MNIST(root, train=True, transform=transform, download=True)
train_ds, val_ds = utils.random_split(train_ds, [50000, 10000], generator=ds_rng)

train_dl = utils.DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl = utils.DataLoader(val_ds, batch_size=128, shuffle=False)

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [6]:
"""
用TorchMetrics统计指标
"""



In [23]:
"""
单论训练:train_one_epoch
"""


def train_one_epoch(
        model: nn.Module,
        dataloader: torch.utils.data.DataLoader[tuple[Tensor, Tensor]],
        loss_fn: nn.Module,
        optimizer: optim.Optimizer,
        metric: Metric,
        device: torch.device,
) -> tuple[float, float]:
    model.train()
    metric.reset()
    total_loss = 0.0

    for X, y in dataloader:
        X = X.to(device)
        y = y.to(device)

        logits = model(X)
        loss = loss_fn(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        metric.update(logits.detach(), y)
    avg_loss = total_loss / len(dataloader)
    avg_metric = metric.compute().item()
    return avg_loss, avg_metric


In [24]:
"""
单轮验证: evaluate
"""


def evaluate(model: nn.Module, dataloader: torch.utils.data.DataLoader[tuple[Tensor, Tensor]], loss_fn: nn.Module,
             metric: Metric, device: torch.device) -> tuple[float, float]:
    model.eval()
    metric.reset()

    total_loss = 0.0

    with  torch.inference_mode():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            logits = model(X)
            loss = loss_fn(logits, y)

            total_loss += loss.item()
            metric.update(logits, y)
    avg_loss = total_loss / len(dataloader)
    avg_metric = metric.compute().item()
    return avg_loss, avg_metric


In [25]:
def train_and_evaluate(
        model: nn.Module,
        train_dl: utils.DataLoader[tuple[Tensor, Tensor]],
        val_dl: utils.DataLoader[tuple[Tensor, Tensor]],
        loss_fn: nn.Module,
        optimizer: optim.Optimizer,
        train_metric: Metric,
        val_metric: Metric,
        metric_name: str,
        num_epochs: int,
        device: torch.device,
) -> None:
    if device is None:
        device = dnnlpy.get_default_device()
    else:
        device = torch.device(device)

    model.to(device)
    train_metric.to(device)
    val_metric.to(device)

    for epoch in range(1, num_epochs + 1):
        loss, score = train_one_epoch(
            model=model,
            dataloader=train_dl,
            loss_fn=loss_fn,
            optimizer=optimizer,
            metric=train_metric,
            device=device,
        )
        val_loss, val_score = evaluate(model=model, dataloader=val_dl, loss_fn=loss_fn, metric=val_metric,
                                       device=device)
        w = len(str(num_epochs))
        print(f"Epoch [{epoch:{w}d}/{num_epochs:{w}d}]"
              f'| loss:{loss:.4f}'
              f'| {metric_name}:{score:.4f}'
              f'|val_loss:{val_loss:.4f}'
              f'|val_{metric_name}:{val_score:.4f}')


torch_rng = set_seed(42)
device = get_default_device()

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

metric = MulticlassAccuracy(num_classes=10)
val_metric = MulticlassAccuracy(num_classes=10)
train_and_evaluate(model=model, train_dl=train_dl, val_dl=val_dl, loss_fn=loss_fn, optimizer=optimizer,
                   train_metric=metric, val_metric=metric, device=device, metric_name='acc', num_epochs=5)


Epoch [1/5]| loss:1.3293| acc:0.7070|val_loss:0.6901|val_acc:0.8458
Epoch [2/5]| loss:0.5447| acc:0.8629|val_loss:0.4661|val_acc:0.8781
Epoch [3/5]| loss:0.4207| acc:0.8855|val_loss:0.4016|val_acc:0.8869
Epoch [4/5]| loss:0.3726| acc:0.8953|val_loss:0.3676|val_acc:0.8976
Epoch [5/5]| loss:0.3451| acc:0.9014|val_loss:0.3495|val_acc:0.8995
